# Q3 · 全20图真实FPGA质量验证
复用已测W2固件与W4a服务，无HDMI。固定A/C2、阈值28；20图×256/512×A/C2共80帧。
运行过程中不要在其他网页提交FPGA任务；每框结束且无报错后再继续，不要Run All。


## 1. 验证下载包并展开
应打印 PACKAGE_BYTES_PASS。只读验证通过后写入新的子文件夹；尚未访问 FPGA。

In [1]:
from pathlib import Path
import hashlib, json, zipfile
if 'browser_package_root' in globals():
    raise RuntimeError('Already extracted; continue to the next cell')
bundle_path = Path('q3_full_quality_r1.zip')
expected_bundle_sha256 = 'be3e5a213c7621837ef622faf367959ed8d193f86c1a0461448f9abcba06e86a'
if hashlib.sha256(bundle_path.read_bytes()).hexdigest() != expected_bundle_sha256:
    raise ValueError('Wrong download bytes; stop before importing or loading FPGA')
with zipfile.ZipFile(bundle_path) as archive:
    members = archive.infolist()
    names = [member.filename for member in members]
    manifest = json.loads(archive.read('q3_manifest.json'))
    if len(names) != len(set(names)) or set(names) != set(manifest['artifacts']) | {'q3_manifest.json'}:
        raise ValueError('Unexpected archive inventory')
    if len(members) > 200 or sum(member.file_size for member in members) > 128 * 1024 * 1024:
        raise ValueError('Unexpected package size')
    for member in members:
        path = Path(member.filename)
        if path.is_absolute() or '..' in path.parts or '\\' in member.filename or ':' in member.filename or member.is_dir():
            raise ValueError('Unsafe package member')
        if member.filename != 'q3_manifest.json':
            data = archive.read(member)
            if {'bytes': len(data), 'sha256': hashlib.sha256(data).hexdigest()} != manifest['artifacts'][member.filename]:
                raise ValueError('Package mismatch: ' + member.filename)
    browser_package_root = (Path.cwd() / 'q3_full_quality_r1').resolve()
    browser_package_root.mkdir(exist_ok=False)
    for member in members:
        destination = browser_package_root / member.filename
        if browser_package_root not in destination.resolve().parents:
            raise ValueError('Package path escaped directory')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
print('PACKAGE_BYTES_PASS: paired W2 and browser application bytes checked; FPGA untouched')


PACKAGE_BYTES_PASS: paired W2 and browser application bytes checked; FPGA untouched


## 2. 检查环境并保留端口
应打印 BROWSER_PRECHECK_PASS 和实际版本。此步不加载FPGA。8090若被占用，保留报错，勿继续。

In [2]:
import sys
if 'browser_context' in globals():
    raise RuntimeError('Already prepared; continue to the next cell')
sys.path.insert(0, str(browser_package_root))
from q3_board import prepare_q3, start, collect_q3, export_q3, finish
browser_context = prepare_q3(browser_package_root, port=8090)


BROWSER_PRECHECK_PASS: paired W2 hashes/runtime/metadata checked; FPGA unchanged
Runtime: {'numpy': '1.21.5', 'Pillow': '9.0.1', 'tornado': '6.1', 'pynq': '3.0.1', 'python': '3.10.4', 'kernel': '5.15.19-xilinx-v2022.1', 'machine': 'armv7l', 'board_clock_utc_unverified': '2022-10-24T07:40:06.625875+00:00', 'runtime_driver_sources': {'DMA': {'path': '/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/lib/dma.py', 'bytes': 26228, 'sha256': 'a7e92fd180d9fa3322e81b49bb4141e08c8f5c8cb685fe608793b46acfe11d49'}, 'AxiGPIO': {'path': '/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/lib/axigpio.py', 'bytes': 10935, 'sha256': 'bb386977825c5c36ee4e5ed814331c1f3105858135fc581362d01e9955d21f27'}, 'allocate': {'path': '/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/buffer.py', 'bytes': 4676, 'sha256': 'c3cfea4e2a94aa9b1981ae79210a69eee0f17b5a5c4cd26e7bc2cf7de01fb236'}, 'Overlay': {'path': '/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlay.py', 'b

## 3. 加载一次并启动服务
应打印 BROWSER_SERVICE_READY。尚未进行图像收发。出现红色报错则保留内核和上下文，不重复加载。

In [3]:
browser_start_status = await start(browser_context)
print(browser_start_status)


BROWSER_SERVICE_READY: W2 A/C2 loaded once; no frame has been sent yet
Open http://192.168.2.99:8090/
Session: 503f7c3f49614e5392568dc260974a6a
{'session_id': '503f7c3f49614e5392568dc260974a6a', 'service_state': 'READY', 'hardware_state': 'IDLE', 'overlay_downloads': 1, 'last_completed_frame_status': 0, 'active_run_id': None, 'pending': 0, 'completed': 0, 'accepted': 0, 'run_limit': 200, 'retained_buffer_count': 0, 'error': None, 'evidence_kind': 'PYNQ_PL_DMA', 'board_acceptance': 'PENDING_INDEPENDENT_VERIFICATION'}


## 4. 自动收集80帧
自动遍历全部20图/两尺寸/A和C2。每完成一帧打印Q3_FRAME_SAVED，最后Q3_COLLECTION_COMPLETE。
期间不要点击应用网页；不用守在这一框逐帧按键。完成表示记录齐全，医学质量和全字匹配须下载后独立复核。


In [4]:
q3_collection = await collect_q3(browser_context)
print(q3_collection)


Q3_FRAME_SAVED: 1/80 im0001 256 A
Q3_FRAME_SAVED: 2/80 im0001 256 C2
Q3_FRAME_SAVED: 3/80 im0001 512 A
Q3_FRAME_SAVED: 4/80 im0001 512 C2
Q3_FRAME_SAVED: 5/80 im0002 256 A
Q3_FRAME_SAVED: 6/80 im0002 256 C2
Q3_FRAME_SAVED: 7/80 im0002 512 A
Q3_FRAME_SAVED: 8/80 im0002 512 C2
Q3_FRAME_SAVED: 9/80 im0003 256 A
Q3_FRAME_SAVED: 10/80 im0003 256 C2
Q3_FRAME_SAVED: 11/80 im0003 512 A
Q3_FRAME_SAVED: 12/80 im0003 512 C2
Q3_FRAME_SAVED: 13/80 im0004 256 A
Q3_FRAME_SAVED: 14/80 im0004 256 C2
Q3_FRAME_SAVED: 15/80 im0004 512 A
Q3_FRAME_SAVED: 16/80 im0004 512 C2
Q3_FRAME_SAVED: 17/80 im0005 256 A
Q3_FRAME_SAVED: 18/80 im0005 256 C2
Q3_FRAME_SAVED: 19/80 im0005 512 A
Q3_FRAME_SAVED: 20/80 im0005 512 C2
Q3_FRAME_SAVED: 21/80 im0044 256 A
Q3_FRAME_SAVED: 22/80 im0044 256 C2
Q3_FRAME_SAVED: 23/80 im0044 512 A
Q3_FRAME_SAVED: 24/80 im0044 512 C2
Q3_FRAME_SAVED: 25/80 im0077 256 A
Q3_FRAME_SAVED: 26/80 im0077 256 C2
Q3_FRAME_SAVED: 27/80 im0077 512 A
Q3_FRAME_SAVED: 28/80 im0077 512 C2
Q3_FRAME_SAVED:

## 5. 浏览器查看两张图并核对状态
第4框完成后，打开 http://192.168.2.99:8090/ 并Ctrl+F5，页眉应为Q3。
固定512/C2/阈值28/最终掩膜/快速无损PNG：选im0002运行并截图，再选im0324运行并截图。
两次都是新PL任务；不要运行20帧循环。两图完成后回来运行下面状态框，应completed=82、READY/IDLE、pending=0、无error/保留缓冲。
这些预览不进入前80帧固定质量队列。单张图的Dice不等于完整留出集质量。


In [5]:
print(browser_context['manager'].status())


{'session_id': '503f7c3f49614e5392568dc260974a6a', 'service_state': 'READY', 'hardware_state': 'IDLE', 'overlay_downloads': 1, 'last_completed_frame_status': 1343570, 'active_run_id': None, 'pending': 0, 'completed': 82, 'accepted': 82, 'run_limit': 200, 'retained_buffer_count': 0, 'error': None, 'evidence_kind': 'PYNQ_PL_DMA', 'board_acceptance': 'PENDING_INDEPENDENT_VERIFICATION'}


## 6. 导出完整记录
应出现Q3_RESULTS_READY。下载本Notebook同目录的q3_results_bundle.zip。不要用网页导出的普通会话包替代，它不包含Q3固定队列记录。


In [6]:
q3_export_info = await export_q3(browser_context, Path.cwd() / 'q3_results_bundle.zip')


BROWSER_RECORDS_READY: /home/xilinx/jupyter_notebooks/retina_q3_quality_r1/q3_full_quality_r1/browser_results/503f7c3f49614e5392568dc260974a6a/exports/browser_session_503f7c3f49614e5392568dc260974a6a.zip
Q3_RESULTS_READY: /home/xilinx/jupyter_notebooks/retina_q3_quality_r1/q3_results_bundle.zip {'bytes': 92221659, 'sha256': '8eb87d0c3c6dfdba411b8f9a0b73e6b37ee3509b1cb321c8c1f85f6e966edccb'}


## 7. 安全结束并下载
应出现BROWSER_SERVICE_CLOSED_IDLE。Ctrl+S，然后File → Download as → Notebook下载q3_quality_start.ipynb。
同时下载q3_results_bundle.zip。若FAULT/红色报错或一分钟无新进展，保留输出与内核，不重复运行/重新加载/释放缓冲。


In [7]:
await finish(browser_context)


BROWSER_SERVICE_CLOSED_IDLE: native buffers already released; original records retained
